# Premier League Polymarket bet analysis

This notebook reads the local SQLite ledger produced by `bet-analytics`. It does not make network requests or need a wallet/API key.

Privacy: work in the Git-ignored `.local.ipynb` copy described in the README. Notebook outputs can contain your public profile history.

In [ ]:
from pathlib import Path
import os
import sys

SEASON = 2026  # 2026 means the 2026/27 season
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (cwd, *cwd.parents) if (candidate / 'pyproject.toml').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Start JupyterLab from the project directory as shown in README.md')
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
matplotlib_config = PROJECT_ROOT / 'data' / 'matplotlib-cache'
matplotlib_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(matplotlib_config))

try:
    import httpx  # required by polymarket_bot
except ModuleNotFoundError as error:
    raise RuntimeError(
        f'Kernel {sys.executable} is missing httpx. Install this project into that environment.'
    ) from error

import matplotlib.pyplot as plt
import pandas as pd

from polymarket_bot.notebook_data import load_notebook_data

DATABASE = PROJECT_ROOT / 'data' / 'bet_history.sqlite3'

data = load_notebook_data(DATABASE, season=SEASON)
bets = data['bets']
trades = data['trades']
closed_positions = data['closed_positions']
xg_matches = data['xg_matches']
address = data['profile_address']
print(f"Profile: {address[:8]}...{address[-4:]}")
print(f"Kernel: {sys.executable}")
print(f"Database: {data['database_path']}")

## Data overview

`bets` contains enriched BUY fills. `trades` is the audit table with every BUY and SELL record. Automatic match results and hypothetical hold P/L apply only to full-time 1X2 selections; other EPL market types remain in the audit data.

In [ ]:
pd.DataFrame(
    {
        'rows': [len(bets), len(trades), len(closed_positions), len(xg_matches)],
    },
    index=['enriched BUY fills', 'all trades', 'closed positions', 'completed xG matches'],
)

In [ ]:
bet_columns = [
    'placed_at', 'fixture', 'market_title', 'market_role', 'selection',
    'entry_price', 'cost_usd',
    'home_form_matches', 'home_form_xgf', 'home_form_xga',
    'away_form_matches', 'away_form_xgf', 'away_form_xga',
    'result', 'hold_pnl_usd', 'post_home_xg', 'post_away_xg',
]
bets[bet_columns].sort_values('placed_at') if not bets.empty else bets

## Audit the imported history

Use this table to compare the stored records with Polymarket before interpreting the results.

In [ ]:
audit_columns = [
    'placed_at', 'side', 'event_slug', 'title', 'outcome',
    'size', 'price', 'usdc_size', 'transaction_hash',
]
trades[audit_columns].sort_values('placed_at') if not trades.empty else trades

## Performance summary

In [ ]:
if bets.empty:
    selections = pd.DataFrame()
    print('No BUY fills are stored for this season.')
else:
    # One selection can contain several fills. Aggregate first so fill count does not
    # masquerade as bet count and entry prices are weighted by shares.
    selections = (
        bets.groupby(
            ['asset', 'event_slug', 'market_role', 'selection'], dropna=False, as_index=False
        )
        .agg(
            placed_at=('placed_at', 'min'),
            kickoff=('kickoff', 'first'),
            fixture=('fixture', 'first'),
            fills=('trade_id', 'count'),
            shares=('shares', 'sum'),
            amount_paid=('cost_usd', 'sum'),
            result=('result', 'first'),
            hold_pnl=('hold_pnl_usd', lambda values: values.sum(min_count=1)),
        )
    )
    selections['weighted_entry'] = selections['amount_paid'] / selections['shares']
    selections['settled_amount_paid'] = selections['amount_paid'].where(
        selections['hold_pnl'].notna()
    )
    summary = (
        selections.groupby(['market_role', 'selection'], dropna=False)
        .agg(
            selections=('asset', 'count'),
            fills=('fills', 'sum'),
            shares=('shares', 'sum'),
            amount_paid=('amount_paid', 'sum'),
            settled_amount_paid=(
                'settled_amount_paid', lambda values: values.sum(min_count=1)
            ),
            hold_pnl=('hold_pnl', lambda values: values.sum(min_count=1)),
        )
        .sort_values('amount_paid', ascending=False)
    )
    summary['weighted_entry'] = summary['amount_paid'] / summary['shares']
    summary['hold_roi'] = summary['hold_pnl'] / summary['settled_amount_paid']
    display(summary)

## Cumulative hold-to-settlement P/L

This is hypothetical for each aggregated selection. It uses Polymarket's recorded USDC cost. The closed-position table remains the source for realized P/L after early sales.

In [ ]:
settled = selections.dropna(subset=['hold_pnl']).sort_values(
    ['kickoff', 'placed_at'], na_position='last'
).copy()
if settled.empty:
    print('No matched bets have settled yet.')
else:
    settled['cumulative_hold_pnl'] = settled['hold_pnl'].cumsum()
    ax = settled.plot(
        x='kickoff', y='cumulative_hold_pnl', marker='o', legend=False, figsize=(10, 4)
    )
    ax.axhline(0, color='black', linewidth=1)
    ax.set(title='Cumulative hold-to-settlement P/L', xlabel='', ylabel='USD')
    plt.tight_layout()
    plt.show()

## Pinnacle benchmark limitation

The live monitor's `PinFair` column is a margin-normalized Pinnacle implied probability. The analytics ledger does **not** save Pinnacle quotes at bet time, so this notebook cannot reconstruct the benchmark that was visible for a historical bet. Preserve a timestamped quote at decision time if you want to audit intuition against that benchmark later.

In [ ]:
selection_columns = [
    'placed_at', 'kickoff', 'fixture', 'market_role', 'selection', 'fills',
    'shares', 'amount_paid', 'weighted_entry', 'result', 'hold_pnl',
]
selections[selection_columns].sort_values('placed_at') if not selections.empty else selections

## Match-level xG source data

In [ ]:
xg_matches[[
    'season', 'kickoff_utc', 'home_team', 'away_team', 'home_goals', 'away_goals',
    'home_xg', 'away_xg', 'source',
]].tail(20) if not xg_matches.empty else xg_matches